In [5]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [6]:
# Connected to Google Drive

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# A. Data Cleaning

##1. Data loading & understanding

Dataset berisi 40.000 baris dan 25 kolom

In [8]:
# Load Data
from pathlib import Path
import pandas as pd

drive_path = Path('/content/drive/MyDrive/WORK WORK/PROJECT - SECURITY ANALYTICS AND MONITORING DASHBOARD/raw_data.csv')

# Read the CSV directly instead of iterating through it as a directory
df = pd.read_csv(drive_path)
df.head()

,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Payload Data,Malware Indicators,Anomaly Scores,Alerts/Warnings,Attack Type,Attack Signature,Action Taken,Severity Level,User Information,Device Information,Network Segment,Geo-location Data,Proxy Information,Firewall Logs,IDS/IPS Alerts,Log Source
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis iusto accusamus ad perferendis esse at. Asperiores ne...,IoC Detected,28.67,NaN,Malware,Known Pattern B,Logged,Low,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT 6.2; Trident/5.0),Segment A,"Jamshedpur, Sikkim",150.9.97.135,Log Data,NaN,Server
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omnis nulla dolore perspiciatis.\nIllo animi mollitia ...,IoC Detected,51.50,NaN,Malware,Known Pattern A,Blocked,Low,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT 6.0; Trident/4.0),Segment B,"Bilaspur, Nagaland",NaN,Log Data,NaN,Firewall
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectus quae nemo ea esse est rerum.,IoC Detected,87.42,Alert Triggered,DDoS,Known Pattern B,Ignored,Low,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.2; Trident/5.0),Segment C,"Bokaro, Rajasthan",114.133.48.179,Log Data,Alert Data,Firewall
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro labore. Minima ab fugit officiis dicta perspiciatis...,NaN,15.79,Alert Triggered,Malware,Known Pattern B,Blocked,Medium,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; rv:1.9.6.20) Gecko/2583-02-14 13:30:10 Firefox/11.0,Segment B,"Jaunpur, Rajasthan",NaN,NaN,Alert Data,Firewall
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi voluptates soluta quis doloribus quas. Iure harum n...,NaN,0.52,Alert Triggered,DDoS,Known Pattern B,Blocked,Low,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT 6.2; Trident/3.0),Segment C,"Anantapur, Tripura",149.6.110.119,NaN,Alert Data,Firewall


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Timestamp               40000 non-null  object 
 1   Source IP Address       40000 non-null  object 
 2   Destination IP Address  40000 non-null  object 
 3   Source Port             40000 non-null  int64  
 4   Destination Port        40000 non-null  int64  
 5   Protocol                40000 non-null  object 
 6   Packet Length           40000 non-null  int64  
 7   Packet Type             40000 non-null  object 
 8   Traffic Type            40000 non-null  object 
 9   Payload Data            40000 non-null  object 
 10  Malware Indicators      20000 non-null  object 
 11  Anomaly Scores          40000 non-null  float64
 12  Alerts/Warnings         19933 non-null  object 
 13  Attack Type             40000 non-null  object 
 14  Attack Signature        40000 non-null

In [12]:
# Preserve the original dataset
df_raw = df.copy(deep=True)

# Create a working copy for data cleaning
df_clean = df_raw.copy(deep=True)

print("Raw dataset shape:", df_raw.shape)
print("Working dataset shape:", df_clean.shape)

Raw dataset shape: (40000, 25)
Working dataset shape: (40000, 25)


##  2. Pemeriksaan kualitas data

In [10]:
df.isnull().sum()

,0
Timestamp,0
Source IP Address,0
Destination IP Address,0
Source Port,0
Destination Port,0
Protocol,0
Packet Length,0
Packet Type,0
Traffic Type,0
Payload Data,0


Kolom	Missing	Persentase

1. Malware Indicators	: 50,00%
2. Alerts/Warnings	: 50,17%
3. Proxy Information	: 49,63%
4. Firewall Logs	: 49,90%
5. IDS/IPS Alerts	: 50,13%





In [13]:

# 1. Duplicate rows
print("Duplicate rows:", df_clean.duplicated().sum())

# 2. Unique values in categorical columns
categorical_cols = [
    "Protocol",
    "Packet Type",
    "Traffic Type",
    "Attack Type",
    "Attack Signature",
    "Action Taken",
    "Severity Level",
    "Network Segment",
    "Log Source"
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False).head(15))

# 3. Numeric value ranges
numeric_cols = [
    "Source Port",
    "Destination Port",
    "Packet Length",
    "Anomaly Scores"
]

for col in numeric_cols:
    if col in df_clean.columns:
        print(f"\n{col}:")
        print(df_clean[col].describe())

# 4. Timestamp format and parseability
timestamp_check = pd.to_datetime(
    df_clean["Timestamp"],
    errors="coerce",
    utc=True
)

print("\nInvalid timestamps:", timestamp_check.isna().sum())
print("Timestamp range:", timestamp_check.min(), "to", timestamp_check.max())

# 5. Check whitespace-only strings
object_cols = df_clean.select_dtypes(include="object").columns

for col in object_cols:
    whitespace_count = df_clean[col].astype("string").str.strip().eq("").sum()
    if whitespace_count > 0:
        print(f"{col}: {whitespace_count} whitespace-only values")


Duplicate rows: 0

Protocol:
Protocol
ICMP    13429
UDP     13299
TCP     13272
Name: count, dtype: int64

Packet Type:
Packet Type
Control    20237
Data       19763
Name: count, dtype: int64

Traffic Type:
Traffic Type
DNS     13376
HTTP    13360
FTP     13264
Name: count, dtype: int64

Attack Type:
Attack Type
DDoS         13428
Malware      13307
Intrusion    13265
Name: count, dtype: int64

Attack Signature:
Attack Signature
Known Pattern A    20076
Known Pattern B    19924
Name: count, dtype: int64

Action Taken:
Action Taken
Blocked    13529
Ignored    13276
Logged     13195
Name: count, dtype: int64

Severity Level:
Severity Level
Medium    13435
High      13382
Low       13183
Name: count, dtype: int64

Network Segment:
Network Segment
Segment C    13408
Segment B    13319
Segment A    13273
Name: count, dtype: int64

Log Source:
Log Source
Firewall    20116
Server      19884
Name: count, dtype: int64

Source Port:
count    40000.000000
mean     32970.356450
std      18560.4256


| Kolom | Temuan | Interpretasi |
|---|---|---|
| `Protocol` | ICMP, UDP, TCP masing-masing sekitar 13 ribu | Tidak ada satu protokol yang sangat mendominasi |
| `Attack Type` | DDoS, Malware, Intrusion masing-masing sekitar 13 ribu | Distribusi jenis serangan cukup merata |
| `Severity Level` | Low, Medium, High masing-masing sekitar 13 ribu | Tidak ada kategori severity yang sangat dominan |
| `Action Taken` | Blocked, Ignored, Logged masing-masing sekitar 13 ribu | Ketiga tindakan muncul dengan frekuensi mirip |
| `Log Source` | Firewall sekitar 20 ribu, Server sekitar 20 ribu | Distribusi sumber log mendekati 50:50 |


### Data Quality Assessment — Initial Findings

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Dataset Size | 40.000 baris, 25 kolom pada data awal | Dataset memiliki 40.000 records untuk dianalisis lebih lanjut. | Pertahankan seluruh records selama belum ditemukan alasan valid untuk menghapusnya. |
| Duplicate Records | 0 baris duplikat persis | Tidak ditemukan baris yang identik pada seluruh kolom yang diperiksa. Namun, duplikasi event secara semantik belum dapat dipastikan. | Tidak perlu menghapus baris berdasarkan exact duplicates pada tahap ini. |
| Categorical Values | Nilai kategori yang ditampilkan konsisten dan memiliki distribusi relatif merata. | Tidak tampak variasi penulisan kategori pada output teratas. Distribusi yang relatif seimbang perlu ditinjau lebih lanjut untuk memahami karakteristik dataset. | Pertahankan label kategori dan periksa konsistensi nilai secara menyeluruh sebelum transformasi. |
| Source and Destination Ports | Source Port: 1.027–65.530; Destination Port: 1.024–65.535 | Seluruh nilai yang diamati berada dalam rentang port numerik yang valid. Konsistensi port dengan jenis trafik dan protokol belum diverifikasi. | Pertahankan nilai port dan lanjutkan validasi antar-kolom. |
| Packet Length | 64–1.500 byte; median 782 byte | Nilai berada dalam rentang yang masuk akal untuk banyak skenario paket jaringan, bergantung pada definisi pengukurannya. | Tidak perlu menghapus nilai ekstrem secara otomatis. |
| Anomaly Scores | 0–100; median 50,345 | Skor berada dalam rentang 0–100. Makna skor dan hubungannya dengan severity atau jenis serangan masih perlu dianalisis. | Pertahankan skor untuk analisis lebih lanjut. |
| Timestamp | 0 timestamp gagal dikonversi; rentang 1 Januari 2020–11 Oktober 2023 | Seluruh nilai berhasil diparse menjadi datetime. Kebenaran zona waktu dan makna timestamp belum diverifikasi. | Pertahankan timestamp dan verifikasi interpretasi zona waktunya sebelum menentukan format akhir untuk Elasticsearch. |
| Missing Values | Lima kolom memiliki sekitar 49,63%–50,17% nilai kosong. | Nilai kosong dalam jumlah besar dapat menunjukkan informasi yang tidak tersedia, tidak berlaku, atau tidak tercatat. Penyebabnya belum diketahui. | Jangan langsung menghapus records atau mengganti nilai kosong sebelum maknanya dipahami. |
| Overall Assessment | Tidak ditemukan exact duplicates dan timestamp yang gagal diparse. | Pemeriksaan awal menunjukkan beberapa aspek kualitas data yang baik, tetapi validitas IP, konsistensi antar-kolom, missing values, dan karakteristik dataset masih perlu diperiksa. | Lanjutkan pemeriksaan kualitas data sebelum melakukan cleaning final dan menyiapkan pipeline. |

In [14]:
# Additional Data Quality Checks


import ipaddress

# 1. Validate source and destination IP addresses
def is_valid_ip(value):
    try:
        ipaddress.ip_address(str(value).strip())
        return True
    except ValueError:
        return False

for col in ["Source IP Address", "Destination IP Address"]:
    if col in df_clean.columns:
        invalid_count = (~df_clean[col].apply(is_valid_ip)).sum()
        print(f"{col} - Invalid IPs: {invalid_count}")

# 2. Check placeholder and whitespace-only values
placeholder_values = [
    "", " ", "unknown", "n/a", "na", "null", "none", "-"
]

for col in df_clean.select_dtypes(include="object").columns:
    values = df_clean[col].astype("string").str.strip().str.lower()
    count = values.isin(placeholder_values).sum()

    if count > 0:
        print(f"{col} - Possible placeholder values: {count}")

# 3. Check numeric boundaries
checks = {
    "Source Port": (0, 65535),
    "Destination Port": (0, 65535),
    "Packet Length": (0, None),
    "Anomaly Scores": (0, 100)
}

for col, (lower, upper) in checks.items():
    if col not in df_clean.columns:
        continue

    invalid = pd.Series(False, index=df_clean.index)

    if lower is not None:
        invalid |= df_clean[col] < lower

    if upper is not None:
        invalid |= df_clean[col] > upper

    print(f"{col} - Values outside configured bounds: {invalid.sum()}")

# 4. Inspect traffic type and protocol combinations
if {"Traffic Type", "Protocol"}.issubset(df_clean.columns):
    display(pd.crosstab(
        df_clean["Traffic Type"],
        df_clean["Protocol"],
        margins=True
    ))


Source IP Address - Invalid IPs: 0
Destination IP Address - Invalid IPs: 0
Source Port - Values outside configured bounds: 0
Destination Port - Values outside configured bounds: 0
Packet Length - Values outside configured bounds: 0
Anomaly Scores - Values outside configured bounds: 0


Protocol,ICMP,TCP,UDP,All
Traffic Type,,,,
DNS,4513,4463,4400,13376
FTP,4386,4373,4505,13264
HTTP,4530,4436,4394,13360
All,13429,13272,13299,40000


| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Source IP Address | 0 alamat IP tidak valid dari 40.000 records | Seluruh alamat IP sumber lolos validasi format IP. Pemeriksaan ini tidak memastikan alamat tersebut aktif atau benar-benar berasal dari aktivitas jaringan nyata. | Tidak diperlukan perbaikan format berdasarkan pemeriksaan ini. |
| Destination IP Address | 0 alamat IP tidak valid dari 40.000 records | Seluruh alamat IP tujuan lolos validasi format IP. | Tidak diperlukan perbaikan format berdasarkan pemeriksaan ini. |
| Source Port | 0 nilai di luar rentang 0–65.535 | Semua nilai memenuhi batas numerik port yang ditetapkan. Konsistensi dengan protokol dan jenis trafik masih perlu diperiksa. | Pertahankan nilai dan lanjutkan validasi semantik. |
| Destination Port | 0 nilai di luar rentang 0–65.535 | Semua nilai memenuhi batas numerik port yang ditetapkan. | Pertahankan nilai dan lanjutkan validasi semantik. |
| Packet Length | 0 nilai di luar batas minimum 0 byte yang ditetapkan | Tidak ditemukan panjang paket negatif. Hasil ini belum membuktikan bahwa semua ukuran paket sesuai dengan konteks jaringan sumber. | Tidak perlu melakukan koreksi berdasarkan pemeriksaan batas ini. |
| Anomaly Scores | 0 nilai di luar rentang 0–100 | Seluruh skor memenuhi rentang yang diasumsikan untuk dataset ini. Makna skor tetap perlu diverifikasi melalui dokumentasi dan analisis. | Pertahankan skor untuk analisis lebih lanjut. |
| Traffic Type vs Protocol | DNS, FTP, dan HTTP masing-masing muncul dengan ICMP, TCP, dan UDP dalam jumlah yang relatif merata. | Kombinasi tersebut patut ditinjau secara semantik. DNS, HTTP, dan FTP lazimnya menggunakan TCP atau UDP, sedangkan ICMP tidak menggunakan port TCP/UDP. Namun, label trafik dan protokol bisa saja didefinisikan secara berbeda dalam dataset ini. | Jangan menghapus records secara otomatis. Periksa dokumentasi dan struktur data sebelum menetapkan aturan validasi antar-kolom. |
| Overall Assessment | Tidak ditemukan IP yang gagal divalidasi maupun nilai numerik di luar batas yang ditetapkan. | Pemeriksaan format dan batas numerik awal menunjukkan hasil yang baik. Validitas semantik antar-kolom masih memerlukan investigasi. | Lanjutkan pemeriksaan kualitas data sebelum melakukan transformasi final. |

In [15]:
# distribusi port dan konsistensi trafik


# 1. Memeriksa distribusi port
port_cols = ["Source Port", "Destination Port"]

for col in port_cols:
    print(f"\n=== {col} ===")
    print("Jumlah nilai unik:", df_clean[col].nunique())
    print("Jumlah port di bawah 1024:", (df_clean[col] < 1024).sum())
    print("Jumlah port di atas 49151:", (df_clean[col] > 49151).sum())
    print("\n20 nilai yang paling sering muncul:")
    print(df_clean[col].value_counts().head(20))

# 2. Memeriksa kombinasi Traffic Type dan Protocol
print("\n=== Traffic Type vs Protocol ===")
display(pd.crosstab(
    df_clean["Traffic Type"],
    df_clean["Protocol"],
    margins=True
))

# 3. Memeriksa distribusi Destination Port berdasarkan Traffic Type
print("\n=== Statistik Destination Port per Traffic Type ===")
display(
    df_clean.groupby("Traffic Type")["Destination Port"]
    .describe()
    .round(2)
)

# 4. Memeriksa port tujuan yang paling sering muncul
print("\n=== Destination Port per Traffic Type ===")
display(
    df_clean.groupby(["Traffic Type", "Destination Port"])
    .size()
    .reset_index(name="Jumlah")
    .sort_values("Jumlah", ascending=False)
    .head(20)
)



=== Source Port ===
Jumlah nilai unik: 29761
Jumlah port di bawah 1024: 0
Jumlah port di atas 49151: 9879

20 nilai yang paling sering muncul:
Source Port
41341    6
34763    5
37135    5
47259    5
53790    5
35304    5
30175    5
60026    5
24528    5
8653     5
17916    5
25710    5
4128     5
4779     5
12179    5
31656    5
16784    5
53397    5
45874    5
15669    5
Name: count, dtype: int64

=== Destination Port ===
Jumlah nilai unik: 29895
Jumlah port di bawah 1024: 0
Jumlah port di atas 49151: 10074

20 nilai yang paling sering muncul:
Destination Port
34117    6
7508     6
59207    5
56765    5
60616    5
36756    5
15150    5
5094     5
25889    5
9565     5
19250    5
31724    5
37248    5
34394    5
38787    5
17896    5
55154    5
52005    5
57516    5
12585    5
Name: count, dtype: int64

=== Traffic Type vs Protocol ===


Protocol,ICMP,TCP,UDP,All
Traffic Type,,,,
DNS,4513,4463,4400,13376
FTP,4386,4373,4505,13264
HTTP,4530,4436,4394,13360
All,13429,13272,13299,40000



=== Statistik Destination Port per Traffic Type ===


,count,mean,std,min,25%,50%,75%,max
Traffic Type,,,,,,,,
DNS,13376.0,33117.29,18558.68,1025.0,17169.00,32878.5,49255.25,65528.0
FTP,13264.0,33415.42,18536.85,1032.0,17416.25,33517.5,49439.25,65534.0
HTTP,13360.0,32921.83,18626.18,1024.0,16689.00,32671.0,49144.25,65535.0



=== Destination Port per Traffic Type ===


,Traffic Type,Destination Port,Jumlah
12922,FTP,5668,4
32836,HTTP,47230,4
34622,HTTP,56672,4
18710,FTP,36756,4
4113,DNS,22754,4
27091,HTTP,16503,4
23246,FTP,60616,4
26682,HTTP,14447,4
29829,HTTP,30804,4
8157,DNS,44330,4


| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Jumlah port unik | Source Port: 29.761; Destination Port: 29.895 | Banyak nilai port berbeda dalam 40.000 baris. | Pertahankan; belum ada alasan untuk menghapus nilai. |
| Port di bawah 1024 | 0 untuk kedua kolom | Tidak ada port dalam rentang port sistem yang umum. | Catat sebagai karakteristik data, bukan otomatis sebagai kesalahan. |
| Port di atas 49151 | Source Port: 9.879; Destination Port: 10.074 | Sekitar seperempat data menggunakan port dalam rentang dinamis/ephemeral. | Pertahankan; nilai masih dalam rentang port yang valid. |
| Frekuensi port teratas | Umumnya muncul 5–6 kali; beberapa muncul 3–4 kali. | Tidak tampak port yang dominan seperti yang sering dijumpai pada layanan umum di banyak log jaringan operasional. | Catat sebagai indikasi pola data yang mungkin sintetis atau acak; perlu validasi lebih lanjut. |
| `Traffic Type` vs `Protocol` | DNS, FTP, dan HTTP muncul hampir merata pada ICMP, TCP, serta UDP. | Kombinasi ini tidak mencerminkan pola protokol layanan jaringan yang lazim. ICMP tidak menggunakan port TCP/UDP. | Jangan menghapus baris; tandai untuk validasi semantik. |
| Statistik port berdasarkan trafik | Rata-rata Destination Port: DNS 33.117,29; FTP 33.415,42; HTTP 32.921,83. | Distribusi port ketiga tipe trafik relatif mirip dan tersebar luas. | Pertahankan sementara; jangan menganggap `Traffic Type` sebagai label layanan yang telah tervalidasi. |

In [16]:

# 1. Menghitung jumlah kolom kosong pada setiap baris
missing_count = df_clean.isna().sum(axis=1)

print("Distribusi jumlah kolom kosong per baris:")
display(
    missing_count.value_counts()
    .sort_index()
    .rename_axis("Jumlah Kolom Kosong")
    .reset_index(name="Jumlah Baris")
)

# 2. Memeriksa pola missing values pada lima kolom utama
missing_cols = [
    "Malware Indicators",
    "Alerts/Warnings",
    "Proxy Information",
    "Firewall Logs",
    "IDS/IPS Alerts"
]

print("\nPola missing values pada lima kolom:")
missing_pattern = df_clean[missing_cols].isna()

display(
    missing_pattern.value_counts()
    .rename("Jumlah Baris")
    .reset_index()
    .sort_values("Jumlah Baris", ascending=False)
    .head(15)
)

# 3. Mengukur hubungan antar-kolom kosong
print("\nKorelasi pola missing values:")
display(missing_pattern.astype(int).corr().round(3))

# 4. Memeriksa missing values berdasarkan Attack Type
print("\nMissing values berdasarkan Attack Type:")
display(
    df_clean.groupby("Attack Type")[missing_cols]
    .apply(lambda group: group.isna().mean().mul(100))
    .round(2)
)

# 5. Memeriksa missing values berdasarkan Severity Level
print("\nMissing values berdasarkan Severity Level:")
display(
    df_clean.groupby("Severity Level")[missing_cols]
    .apply(lambda group: group.isna().mean().mul(100))
    .round(2)
)


Distribusi jumlah kolom kosong per baris:


,Jumlah Kolom Kosong,Jumlah Baris
0,0,1237
1,1,6285
2,2,12552
3,3,12395
4,4,6300
5,5,1231



Pola missing values pada lima kolom:


,Malware Indicators,Alerts/Warnings,Proxy Information,Firewall Logs,IDS/IPS Alerts,Jumlah Baris
0,False,True,False,False,False,1309
1,False,True,True,False,False,1306
2,True,False,False,False,True,1306
3,False,True,False,True,True,1296
4,True,True,True,False,True,1293
5,True,True,False,True,True,1279
6,True,False,False,True,True,1277
7,False,False,False,False,True,1275
8,False,True,False,True,False,1268
9,False,True,True,True,True,1266



Korelasi pola missing values:


,Malware Indicators,Alerts/Warnings,Proxy Information,Firewall Logs,IDS/IPS Alerts
Malware Indicators,1.000,-0.001,0.001,0.001,0.007
Alerts/Warnings,-0.001,1.000,-0.004,-0.001,-0.003
Proxy Information,0.001,-0.004,1.000,-0.001,-0.005
Firewall Logs,0.001,-0.001,-0.001,1.000,0.006
IDS/IPS Alerts,0.007,-0.003,-0.005,0.006,1.000



Missing values berdasarkan Attack Type:


,Malware Indicators,Alerts/Warnings,Proxy Information,Firewall Logs,IDS/IPS Alerts
Attack Type,,,,,
DDoS,50.45,50.31,49.47,50.01,50.13
Intrusion,49.75,50.81,50.27,50.07,50.34
Malware,49.79,49.39,49.15,49.62,49.91



Missing values berdasarkan Severity Level:


,Malware Indicators,Alerts/Warnings,Proxy Information,Firewall Logs,IDS/IPS Alerts
Severity Level,,,,,
High,49.96,50.07,50.18,49.96,49.75
Low,50.19,50.17,49.69,50.04,50.08
Medium,49.85,50.26,49.01,49.71,50.54


Interpretasi Hasil:

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Jumlah kolom kosong per baris | Semua 40.000 baris memiliki 0–5 nilai kosong. | Missing values terdapat pada lima kolom tertentu dan jumlahnya bervariasi pada setiap baris. | Pertahankan seluruh baris untuk sementara. |
| Pola missing values | Kombinasi nilai kosong tersebar dalam berbagai pola. | Tidak terlihat satu pola kombinasi yang mendominasi secara ekstrem di antara 15 pola teratas yang ditampilkan. | Jangan mengisi semua nilai kosong dengan satu nilai secara otomatis. |
| Korelasi missing values | Korelasi antarkolom berkisar sekitar -0,005 hingga 0,007. | Tidak tampak hubungan linear yang berarti antara status kosong pada kelima kolom tersebut. | Pertahankan nilai kosong sementara; korelasi mendekati nol bukan bukti bahwa missing values sepenuhnya acak. |
| Missing values berdasarkan Attack Type | Persentase missing values pada setiap kolom sekitar 49–51% untuk DDoS, Intrusion, dan Malware. | Tingkat missing values relatif serupa pada setiap kategori serangan. | Belum ada alasan untuk mengisi atau menghapus nilai berdasarkan Attack Type. |
| Missing values berdasarkan Severity Level | Persentase missing values pada setiap kolom sekitar 49–51% untuk High, Low, dan Medium. | Tingkat missing values relatif serupa pada setiap kategori severity. | Belum ada alasan untuk menangani missing values secara berbeda berdasarkan Severity Level. |

### Data Source & Semantic Validation

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Sumber awal yang diberikan | Notebook Kaggle `cyberattack-eda-ml` | Tautan merujuk ke notebook analisis, bukan halaman dataset asli. | Gunakan sebagai referensi analisis, bukan satu-satunya sumber dokumentasi. |
| Sumber dataset | GitHub `incribo-inc/cybersecurity_attacks` dan salinan di Hugging Face | Nama dataset, jumlah baris, dan struktur 25 kolom cocok dengan dataset yang dianalisis. | Catat sebagai sumber yang cocok secara struktur; hubungan langsung dengan file yang diunduh masih perlu dikonfirmasi. |
| Asal-usul data | Deskripsi publik menyebut dataset sintetis dari Incribo. | Ada indikasi kuat bahwa dataset dibuat untuk simulasi atau pembelajaran, bukan dikonfirmasi sebagai log produksi. | Jangan mengklaim bahwa dataset merepresentasikan serangan siber nyata tanpa bukti tambahan. |
| Konsistensi `Traffic Type` dan `Protocol` | DNS, FTP, dan HTTP muncul pada ICMP, TCP, dan UDP. | Pola ini tidak sesuai dengan hubungan protokol layanan yang lazim. | Pertahankan data, tetapi dokumentasikan sebagai keterbatasan semantik. |
| Missing values | Lima kolom memiliki sekitar 50% nilai kosong. | Makna nilai kosong belum terverifikasi. | Jangan mengubah missing values menjadi “tidak ada ancaman” tanpa bukti. |
| Kelayakan proyek | Memiliki timestamp, IP, port, kategori serangan, severity, dan sumber log. | Struktur mendukung latihan data cleaning, analisis, pipeline, dan dashboard. | Lanjutkan proyek dengan keterbatasan dataset yang dinyatakan secara transparan. |

In [17]:
# Review Variasi Kolom

cols_to_review = [
    "Malware Indicators",
    "Alerts/Warnings",
    "Firewall Logs",
    "IDS/IPS Alerts"
]

for col in cols_to_review:
    print(f"\n=== {col} ===")
    print(df_clean[col].value_counts(dropna=False).head(10))


=== Malware Indicators ===
Malware Indicators
IoC Detected    20000
NaN             20000
Name: count, dtype: int64

=== Alerts/Warnings ===
Alerts/Warnings
NaN                20067
Alert Triggered    19933
Name: count, dtype: int64

=== Firewall Logs ===
Firewall Logs
Log Data    20039
NaN         19961
Name: count, dtype: int64

=== IDS/IPS Alerts ===
IDS/IPS Alerts
NaN           20050
Alert Data    19950
Name: count, dtype: int64


### Validasi Nilai pada Kolom Indikator dan Log

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Malware Indicators | `IoC Detected`: 20.000; kosong: 20.000 | Seluruh nilai yang tersedia hanya menyatakan `IoC Detected`. Tidak ada variasi indikator spesifik yang dapat dianalisis. | Pertahankan sementara; jangan menganggap nilai kosong berarti tidak ada malware. |
| Alerts/Warnings | `Alert Triggered`: 19.933; kosong: 20.067 | Kolom hanya memiliki satu kategori alert nonkosong, dengan proporsi kosong sekitar 50,17%. | Pertahankan sementara; makna nilai kosong belum terverifikasi. |
| Firewall Logs | `Log Data`: 20.039; kosong: 19.961 | Nilai nonkosong hanya berupa label generik, bukan isi log firewall yang terperinci. | Tidak cocok untuk analisis isi log; pertimbangkan untuk mengecualikannya dari analisis utama. |
| IDS/IPS Alerts | `Alert Data`: 19.950; kosong: 20.050 | Nilai nonkosong hanya berupa label generik, bukan rincian alert IDS/IPS. | Tidak cocok untuk analisis isi alert; pertimbangkan untuk mengecualikannya dari analisis utama. |
| Kualitas informasi | Empat kolom memiliki satu nilai nonkosong yang berulang. | Variasi informasinya sangat terbatas sehingga manfaatnya untuk analisis eksploratif rendah. | Evaluasi relevansi kolom berdasarkan tujuan dashboard; simpan dataset mentah sebagai referensi. |

In [18]:
# Evaluasi Kolom


cols_to_review = [
    "Proxy Information",
    "Payload Data",
    "User Information",
    "Device Information",
    "Geo-location Data"
]

for col in cols_to_review:
    print(f"\n=== {col} ===")
    print("Jumlah nilai kosong:", df_clean[col].isna().sum())
    print("Jumlah nilai unik:", df_clean[col].nunique(dropna=True))
    print("\n5 nilai yang paling sering muncul:")
    print(df_clean[col].value_counts(dropna=False).head(5))



=== Proxy Information ===
Jumlah nilai kosong: 19851
Jumlah nilai unik: 20148

5 nilai yang paling sering muncul:
Proxy Information
NaN                19851
39.123.165.122         2
211.195.173.251        1
211.232.254.112        1
104.119.37.246         1
Name: count, dtype: int64

=== Payload Data ===
Jumlah nilai kosong: 0
Jumlah nilai unik: 40000

5 nilai yang paling sering muncul:
Payload Data
Fugiat tenetur natus perferendis. Mollitia blanditiis numquam enim nam repellat magnam. Delectus ex vel.                                                                                                    1
Consequatur repudiandae quo omnis consequuntur omnis maxime.\nDolorum mollitia impedit eveniet enim distinctio dolorem. Facilis ullam voluptas accusantium commodi. Dolore nisi eum accusantium enim.       1
Fugiat dolorem nesciunt sint et. Esse eveniet dolores totam quia perferendis. Possimus dignissimos voluptate optio aliquid totam nobis.\nOdit laborum repudiandae quaerat nostrum repudia

### Validasi Proxy, Payload, User, Device, dan Geo-location

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
|---|---|---|---|
| Proxy Information | 19.851 kosong; 20.148 nilai unik dari 20.149 nilai nonkosong. | Hampir setiap nilai nonkosong berbeda dan menyerupai alamat IP. Belum diketahui apakah alamat tersebut benar-benar merepresentasikan proxy yang digunakan. | Pertahankan sementara; evaluasi relevansi untuk analisis utama. |
| Payload Data | 40.000 nilai unik; tidak ada nilai kosong. Contoh berupa kalimat bahasa Inggris generik. | Teks tidak tampak seperti payload jaringan teknis yang terstruktur. Kemungkinan merupakan teks sintetis atau placeholder, tetapi perlu konfirmasi dari dokumentasi. | Jangan gunakan untuk analisis konten serangan; pertimbangkan mengecualikan dari indeks utama jika tidak diperlukan. |
| User Information | 32.389 nilai unik; tidak ada nilai kosong. Nilai tampak berupa nama orang. | Berpotensi berupa data identitas sintetis atau sensitif. Nama yang tampak nyata tidak membuktikan bahwa data tersebut benar-benar berasal dari orang nyata. | Hindari penggunaan untuk analisis identitas; pertimbangkan mengecualikan dari indeks analitik utama. |
| Device Information | 32.104 nilai unik; tidak ada nilai kosong. Contoh berupa user-agent string. | Dapat mendukung pengelompokan berdasarkan browser atau sistem operasi, tetapi validitas dan representativitasnya belum diketahui. | Pertahankan untuk eksplorasi; evaluasi kualitas format sebelum analisis lebih lanjut. |
| Geo-location Data | 8.723 nilai unik; tidak ada nilai kosong. Beberapa pasangan nama kota dan negara bagian tampak tidak lazim. | Konsistensi geografis belum tervalidasi; lokasi tidak boleh langsung dianggap akurat. | Pertahankan sementara; hindari klaim geografis sampai validitasnya diperiksa. |

## Rekomendasi Kolom

### Rekomendasi Kolom untuk Analisis Utama

| Kelompok | Kolom | Rekomendasi |
| :--- | :--- | :--- |
| **Waktu dan jaringan** | Timestamp, Source IP Address, Destination IP Address, Source Port, Destination Port, Protocol, Packet Length, Network Segment, Log Source | Pertahankan sebagai fondasi analisis jaringan. |
| **Kategori dan penilaian** | Traffic Type, Attack Type, Attack Signature, Action Taken, Severity Level, Anomaly Scores, Packet Type | Pertahankan, dengan catatan label belum tentu tervalidasi secara operasional. |
| **Indikator dan alert** | Malware Indicators, Alerts/Warnings, Firewall Logs, IDS/IPS Alerts | Pertahankan di data mentah; nilai informasinya terbatas untuk analisis terperinci. |
| **Informasi tambahan** | Proxy Information, Device Information, Geo-location Data | Pertahankan sementara dan gunakan hanya jika analisis yang direncanakan memang memerlukannya. |
| **Teks dan identitas** | Payload Data, User Information | Pertimbangkan untuk tidak memasukkannya ke indeks analitik utama, kecuali ada kebutuhan yang jelas. |

### Rekomendasi Keputusan untuk 25 Kolom

| Kolom | Keputusan sementara | Alasan |
| :--- | :--- | :--- |
| **Timestamp** | Konversi ke datetime UTC | Penting untuk analisis tren dan monitoring berdasarkan waktu. |
| **Source IP Address** | Pertahankan | Identifikasi sumber koneksi; format IP valid. |
| **Destination IP Address** | Pertahankan | Identifikasi tujuan koneksi; format IP valid. |
| **Source Port** | Pertahankan sebagai integer | Nilai berada dalam rentang valid. |
| **Destination Port** | Pertahankan sebagai integer | Nilai berada dalam rentang valid; distribusinya tampak acak sehingga perlu kehati-hatian saat interpretasi. |
| **Protocol** | Pertahankan | Berguna untuk analisis protokol jaringan. |
| **Packet Length** | Pertahankan sebagai numerik | Dapat dipakai untuk analisis distribusi ukuran paket. |
| **Packet Type** | Pertahankan | Kategori paket dapat digunakan untuk segmentasi. |
| **Traffic Type** | Pertahankan | Berguna untuk analisis tipe trafik, tetapi relasinya dengan Protocol perlu ditafsirkan hati-hati. |
| **Payload Data** | Simpan di raw; tidak diprioritaskan untuk index utama | Isinya tampak seperti kalimat generik, bukan payload jaringan yang dapat diandalkan. |
| **Malware Indicators** | Pertahankan nilai asli dan missing | Nilai nonkosong hanya berupa label generik IoC Detected; jangan mengisi missing tanpa bukti. |
| **Anomaly Scores** | Pertahankan sebagai numerik | Rentangnya 0–100 dan dapat digunakan untuk analisis skor anomali. |
| **Alerts/Warnings** | Pertahankan nilai asli dan missing | Nilai nonkosong hanya Alert Triggered; makna missing belum pasti. |
| **Attack Type** | Pertahankan | Salah satu kategori utama untuk analisis keamanan. |
| **Attack Signature** | Pertahankan | Dapat dipakai untuk segmentasi, tetapi hanya memiliki dua kategori yang hampir seimbang. |
| **Action Taken** | Pertahankan | Berguna untuk melihat tindakan seperti Blocked, Ignored, dan Logged. |
| **Severity Level** | Pertahankan | Berguna untuk memfilter dan membandingkan tingkat keparahan. |
| **User Information** | Simpan di raw; keluarkan dari index utama untuk sementara | Berisi nama-nama; tidak dibutuhkan untuk analisis jaringan utama dan berpotensi memuat informasi identitas. |
| **Device Information** | Pertahankan sebagai kolom tambahan | Berisi string mirip user-agent; bisa berguna untuk eksplorasi perangkat, tetapi validitasnya belum dipastikan. |
| **Network Segment** | Pertahankan | Berguna untuk membandingkan trafik antarsemen jaringan. |
| **Geo-location Data** | Pertahankan, tetapi jangan jadikan dasar analisis geografis dulu | Pasangan lokasi tampak perlu divalidasi. |
| **Proxy Information** | Pertahankan nilai asli dan missing | Banyak nilai menyerupai IP, tetapi makna dan keandalannya belum terverifikasi. |
| **Firewall Logs** | Pertahankan nilai asli dan missing | Nilai nonkosong hanya label generik Log Data; informasi detail terbatas. |
| **IDS/IPS Alerts** | Pertahankan nilai asli dan missing | Nilai nonkosong hanya label generik Alert Data; informasi detail terbatas. |
| **Log Source** | Pertahankan | Berguna untuk membedakan sumber log, seperti Firewall dan Server. |

In [ ]:
df.head()

,Protocol,Packet Length,Packet Type,Traffic Type,Malware Indicators,Anomaly Scores,Alerts/Warnings,Attack Type,Attack Signature,Action Taken,Severity Level,Network Segment
0,ICMP,503,Data,HTTP,IoC Detected,28.67,No Alert,Malware,Known Pattern B,Logged,Low,Segment A
1,ICMP,1174,Data,HTTP,IoC Detected,51.50,No Alert,Malware,Known Pattern A,Blocked,Low,Segment B
2,UDP,306,Control,HTTP,IoC Detected,87.42,Alert Triggered,DDoS,Known Pattern B,Ignored,Low,Segment C
3,UDP,385,Data,HTTP,None Detected,15.79,Alert Triggered,Malware,Known Pattern B,Blocked,Medium,Segment B
4,TCP,1462,Data,DNS,None Detected,0.52,Alert Triggered,DDoS,Known Pattern B,Blocked,Low,Segment C


## 2. Standardisasi Tipe Data dan Teks

In [19]:
# Check current data types
print("Current data types:")
display(df_clean.dtypes)

print("\nTimestamp sample:")
display(df_clean[["Timestamp"]].head())

print("\nCurrent dataset shape:")
print(df_clean.shape)

Current data types:


,0
Timestamp,object
Source IP Address,object
Destination IP Address,object
Source Port,int64
Destination Port,int64
Protocol,object
Packet Length,int64
Packet Type,object
Traffic Type,object
Payload Data,object



Timestamp sample:


,Timestamp
0,2023-05-30 06:33:58
1,2020-08-26 07:08:30
2,2022-11-13 08:23:25
3,2023-07-02 10:38:46
4,2023-07-16 13:11:07



Current dataset shape:
(40000, 25)


Maka Keputusan Selanjutnya:

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
| :--- | :--- | :--- | :--- |
| **Ukuran dataset** | 40.000 baris, 25 kolom | Seluruh kolom masih tersedia untuk proses analisis. | Pertahankan dataset kerja. |
| **Timestamp** | object | Waktu masih disimpan sebagai teks, belum sebagai tipe datetime pandas. | Konversi ke datetime. |
| **Kolom port** | int64 | Kedua kolom port sudah bertipe bilangan bulat. | Pertahankan. |
| **Packet Length** | int64 | Panjang paket sudah bertipe bilangan bulat. | Pertahankan. |
| **Anomaly Scores** | float64 | Skor anomali sudah bertipe numerik desimal. | Pertahankan. |
| **Kolom lainnya** | object | Umumnya berupa teks atau kategori; tipe ini belum tentu salah. | Periksa sesuai kebutuhan masing-masing kolom. |

In [20]:
# Convert Timestamp to datetime in UTC
df_clean["Timestamp"] = pd.to_datetime(
    df_clean["Timestamp"],
    errors="coerce",
    utc=True
)

# Validate the conversion
print("Timestamp dtype:", df_clean["Timestamp"].dtype)
print("Invalid timestamps:", df_clean["Timestamp"].isna().sum())
print("Earliest timestamp:", df_clean["Timestamp"].min())
print("Latest timestamp:", df_clean["Timestamp"].max())

display(df_clean[["Timestamp"]].head())

Timestamp dtype: datetime64[ns, UTC]
Invalid timestamps: 0
Earliest timestamp: 2020-01-01 00:43:27+00:00
Latest timestamp: 2023-10-11 19:34:23+00:00


,Timestamp
0,2023-05-30 06:33:58+00:00
1,2020-08-26 07:08:30+00:00
2,2022-11-13 08:23:25+00:00
3,2023-07-02 10:38:46+00:00
4,2023-07-16 13:11:07+00:00


Seluruh 40.000 nilai Kolom Timestamp berhasil dikonversi tanpa menghasilkan NaT

In [21]:
# Memeriksa Konsistensi Teks

# Identify text columns
text_cols = df_clean.select_dtypes(include=["object"]).columns

# Count values with leading or trailing whitespace
whitespace_summary = {}

for col in text_cols:
    values = df_clean[col].dropna().astype(str)
    whitespace_summary[col] = values.ne(values.str.strip()).sum()

whitespace_summary = pd.Series(
    whitespace_summary,
    name="Values with Leading/Trailing Whitespace"
)

display(
    whitespace_summary[whitespace_summary > 0]
    .sort_values(ascending=False)
    .to_frame()
)

if whitespace_summary.sum() == 0:
    print("No leading/trailing whitespace found in text columns.")

,Values with Leading/Trailing Whitespace


No leading/trailing whitespace found in text columns.


Tidak ditemukan masalah whitespace pada kolom teks (tidak ada nilai teks dengan spasi berlebih) berdasarkan pemeriksaan ini. Data dipertahankan tanpa perubahan tambahan.

In [22]:
# Memeriksa Variasi Kategori

# Inspect unique categorical values
categorical_cols = [
    "Protocol",
    "Packet Type",
    "Traffic Type",
    "Attack Type",
    "Attack Signature",
    "Action Taken",
    "Severity Level",
    "Network Segment",
    "Log Source"
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print("Unique values:", df_clean[col].nunique())
    print(df_clean[col].value_counts(dropna=False))


=== Protocol ===
Unique values: 3
Protocol
ICMP    13429
UDP     13299
TCP     13272
Name: count, dtype: int64

=== Packet Type ===
Unique values: 2
Packet Type
Control    20237
Data       19763
Name: count, dtype: int64

=== Traffic Type ===
Unique values: 3
Traffic Type
DNS     13376
HTTP    13360
FTP     13264
Name: count, dtype: int64

=== Attack Type ===
Unique values: 3
Attack Type
DDoS         13428
Malware      13307
Intrusion    13265
Name: count, dtype: int64

=== Attack Signature ===
Unique values: 2
Attack Signature
Known Pattern A    20076
Known Pattern B    19924
Name: count, dtype: int64

=== Action Taken ===
Unique values: 3
Action Taken
Blocked    13529
Ignored    13276
Logged     13195
Name: count, dtype: int64

=== Severity Level ===
Unique values: 3
Severity Level
Medium    13435
High      13382
Low       13183
Name: count, dtype: int64

=== Network Segment ===
Unique values: 3
Network Segment
Segment C    13408
Segment B    13319
Segment A    13273
Name: count, dt

Tidak ditemukan indikasi masalah konsistensi penulisan pada sembilan kolom kategorikal yang diperiksa. Tidak diperlukan standardisasi kategori tambahan pada tahap ini. Distribusi yang relatif seimbang dan validitas makna kategori akan dievaluasi secara terpisah.

In [23]:
# Memeriksa Konsistensi Nilai Teks Lainnya

# Check common missing-value placeholders in text columns
placeholders = ["", "unknown", "n/a", "na", "null", "none", "-"]

text_cols = df_clean.select_dtypes(include=["object"]).columns

placeholder_results = []

for col in text_cols:
    values = df_clean[col].dropna().astype(str).str.strip().str.lower()

    for placeholder in placeholders:
        count = values.eq(placeholder).sum()

        if count > 0:
            placeholder_results.append({
                "Column": col,
                "Placeholder": placeholder if placeholder else "(empty string)",
                "Count": int(count)
            })

if placeholder_results:
    display(pd.DataFrame(placeholder_results))
else:
    print("No common missing-value placeholders found.")

No common missing-value placeholders found.


Pemeriksaan placeholder umum tidak menemukan nilai yang cocok dengan daftar pemeriksaan. Hasil ini digunakan sebagai pemeriksaan tambahan, bukan bukti bahwa semua nilai kosong tersamarkan telah teridentifikasi.

## 3. Penanganan Missing Value

In [25]:
# Summarize missing values
missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage (%)": (
        df_clean.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(
    "Missing Percentage (%)",
    ascending=False
)

display(missing_summary)

,Missing Count,Missing Percentage (%)
Alerts/Warnings,20067,50.17
IDS/IPS Alerts,20050,50.12
Malware Indicators,20000,50.00
Firewall Logs,19961,49.90
Proxy Information,19851,49.63


### Missing Values Assessment

| Aspek Pemeriksaan | Hasil | Interpretasi | Keputusan |
| :--- | :--- | :--- | :--- |
| **Alerts/Warnings** | 20.067 (50,17%) missing | Sekitar setengah baris tidak memiliki nilai pada kolom ini. Nilai kosong belum dapat dipastikan berarti tidak ada alert. | Pertahankan missing values. |
| **IDS/IPS Alerts** | 20.050 (50,12%) missing | Informasi alert IDS/IPS tidak tersedia pada sekitar setengah baris. | Pertahankan missing values. |
| **Malware Indicators** | 20.000 (50,00%) missing | Nilai kosong tidak dapat langsung diartikan sebagai tidak adanya indikator malware. | Pertahankan missing values. |
| **Firewall Logs** | 19.961 (49,90%) missing | Informasi firewall tidak tersedia pada sekitar setengah baris; makna nilai kosong belum diketahui. | Pertahankan missing values. |
| **Proxy Information** | 19.851 (49,63%) missing | Informasi proxy tidak tersedia pada sekitar setengah baris dan makna missing belum terverifikasi. | Pertahankan missing values. |


**Kesimpulan:** Kelima kolom memiliki missing values dalam proporsi tinggi, sekitar 49,63–50,17%. Karena makna nilai kosong belum dapat dipastikan, missing values dipertahankan tanpa imputasi agar tidak menambahkan informasi yang belum terverifikasi.

In [26]:
# Validasi Tipe Data & Nilai Numerik

# Check missing values in numeric columns
numeric_cols = df_clean.select_dtypes(include="number").columns

numeric_missing = df_clean[numeric_cols].isna().sum()

print("Missing values in numeric columns:")
display(numeric_missing.to_frame(name="Missing Count"))

print("\nNumeric data types:")
display(df_clean[numeric_cols].dtypes.to_frame(name="Data Type"))

Missing values in numeric columns:


,Missing Count
Source Port,0
Destination Port,0
Packet Length,0
Anomaly Scores,0



Numeric data types:


,Data Type
Source Port,int64
Destination Port,int64
Packet Length,int64
Anomaly Scores,float64


Keempat kolom numerik tidak memiliki missing values dan tipe datanya sesuai untuk analisis numerik.

In [27]:
# Final check for row count and Timestamp validity
print("Dataset shape:", df_clean.shape)
print("Missing Timestamp:", df_clean["Timestamp"].isna().sum())
print("Duplicate rows:", df_clean.duplicated().sum())

Dataset shape: (40000, 25)
Missing Timestamp: 0
Duplicate rows: 0


Dataset mempertahankan ukuran awalnya, seluruh timestamp valid setelah konversi, dan tidak ditemukan duplikat. Pemeriksaan kualitas dasar telah selesai. Missing values pada lima kolom teks tetap dipertahankan karena maknanya belum terverifikasi.

In [28]:
# Memeriksa Raw Dataset VS Clean Dataset

print("Raw dataset shape:", df_raw.shape)
print("Clean dataset shape:", df_clean.shape)

print("Raw columns:", len(df_raw.columns))
print("Clean columns:", len(df_clean.columns))

Raw dataset shape: (40000, 25)
Clean dataset shape: (40000, 25)
Raw columns: 25
Clean columns: 25


In [29]:
# Validasi Perubahan

# Check whether non-Timestamp columns remain unchanged
other_cols = [
    col for col in df_raw.columns
    if col != "Timestamp"
]

unchanged_cols = all(
    df_raw[col].equals(df_clean[col])
    for col in other_cols
)

print("Non-Timestamp columns unchanged:", unchanged_cols)
print("Raw Timestamp dtype:", df_raw["Timestamp"].dtype)
print("Clean Timestamp dtype:", df_clean["Timestamp"].dtype)
print("Clean Timestamp missing:", df_clean["Timestamp"].isna().sum())

Non-Timestamp columns unchanged: True
Raw Timestamp dtype: object
Clean Timestamp dtype: datetime64[ns, UTC]
Clean Timestamp missing: 0


Validasi menunjukkan bahwa perubahan pada tahap ini hanya terjadi pada kolom Timestamp. Seluruh kolom lainnya tetap identik dengan dataset asli, sehingga proses cleaning dapat dilanjutkan.

### Proses, Status & Hasil

| Proses | Status | Hasil |
| :--- | :--- | :--- |
| **Memeriksa struktur dataset** | Selesai | 40.000 baris, 25 kolom |
| **Memeriksa duplikat** | Selesai | Tidak ditemukan duplikat |
| **Memvalidasi IP address** | Selesai | Format IP valid |
| **Memeriksa rentang numerik** | Selesai | Nilai memenuhi rentang yang diperiksa |
| **Mengonversi Timestamp** | Selesai | datetime64[ns, UTC], 0 nilai tidak valid |
| **Memeriksa konsistensi teks dan kategori** | Selesai | Tidak ditemukan masalah penulisan yang teridentifikasi dari pemeriksaan tersebut |
| **Menganalisis missing values** | Selesai | Lima kolom memiliki sekitar 50% missing; nilai kosong dipertahankan |
| **Memvalidasi perubahan cleaning** | Selesai | Kolom selain Timestamp tidak berubah |

# B. Menetapkan skema index Elasticsearch

Membuat df_index yang berisi 16 kolom kandidat untuk dashboard. Dataset df_raw dan df_clean tidak akan diubah.

Kolom yang dipilih:
- Waktu dan jaringan: Timestamp, Source IP Address, Destination IP Address, Source Port, Destination Port, Protocol, Packet Length
- Analitik keamanan: Attack Type, Severity Level, Action Taken, Anomaly Scores, Attack Signature
- Konteks trafik: Packet Type, Traffic Type, Network Segment, Log Source

In [30]:
# Define columns for the main Elasticsearch index
index_columns = [
    "Timestamp",
    "Source IP Address",
    "Destination IP Address",
    "Source Port",
    "Destination Port",
    "Protocol",
    "Packet Length",
    "Attack Type",
    "Severity Level",
    "Action Taken",
    "Anomaly Scores",
    "Attack Signature",
    "Packet Type",
    "Traffic Type",
    "Network Segment",
    "Log Source"
]

# Create a separate dataset for Elasticsearch
df_index = df_clean[index_columns].copy()

# Validate the new dataset
print("Index dataset shape:", df_index.shape)
print("Original clean dataset shape:", df_clean.shape)
print("Missing values in index dataset:", df_index.isna().sum().sum())

display(df_index.head())

Index dataset shape: (40000, 16)
Original clean dataset shape: (40000, 25)
Missing values in index dataset: 0


,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Attack Type,Severity Level,Action Taken,Anomaly Scores,Attack Signature,Packet Type,Traffic Type,Network Segment,Log Source
0,2023-05-30 06:33:58+00:00,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Malware,Low,Logged,28.67,Known Pattern B,Data,HTTP,Segment A,Server
1,2020-08-26 07:08:30+00:00,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Malware,Low,Blocked,51.50,Known Pattern A,Data,HTTP,Segment B,Firewall
2,2022-11-13 08:23:25+00:00,63.79.210.48,198.219.82.17,16811,53600,UDP,306,DDoS,Low,Ignored,87.42,Known Pattern B,Control,HTTP,Segment C,Firewall
3,2023-07-02 10:38:46+00:00,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Malware,Medium,Blocked,15.79,Known Pattern B,Data,HTTP,Segment B,Firewall
4,2023-07-16 13:11:07+00:00,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,DDoS,Low,Blocked,0.52,Known Pattern B,Data,DNS,Segment C,Firewall


Maka Selanjutnya:

| Aspek pemeriksaan | Hasil | Interpretasi | Keputusan |
| :--- | :--- | :--- | :--- |
| **Jumlah baris** | 40.000 | Semua event masih dipertahankan. | Lanjut |
| **Jumlah kolom** | 16 dari 25 | Memilih kolom yang relevan untuk dashboard monitoring. | Lanjut |
| **Missing values** | 0 | Tidak ada nilai kosong pada 16 kolom terpilih. | Lanjut |
| **Timestamp** | Menampilkan zona waktu +00:00 | Timestamp sudah dikonversi menjadi datetime UTC. | Petakan sebagai date |
| **Source/Destination IP** | Berformat alamat IP | Cocok dipetakan sebagai tipe ip di Elasticsearch. | Lanjut |
| **Port dan Packet Length** | Berupa angka bulat | Cocok dipetakan sebagai integer. | Lanjut |
| **Anomaly Scores** | Angka desimal | Cocok dipetakan sebagai float. | Lanjut |
| **Kolom kategori** | Berisi label seperti Malware, Low, dan Blocked | Cocok untuk filter, agregasi, dan visualisasi Kibana. | Petakan sebagai keyword |

In [31]:
# Cek Tipe Data untuk Mapping

# Periksa tipe data dataset yang akan diindeks
print("Tipe data df_index:")
display(df_index.dtypes.to_frame(name="Data Type"))

print("\nJumlah nilai unik per kolom:")
display(df_index.nunique().to_frame(name="Unique Values"))

Tipe data df_index:


,Data Type
Timestamp,"datetime64[ns, UTC]"
Source IP Address,object
Destination IP Address,object
Source Port,int64
Destination Port,int64
Protocol,object
Packet Length,int64
Attack Type,object
Severity Level,object
Action Taken,object



Jumlah nilai unik per kolom:


,Unique Values
Timestamp,39997
Source IP Address,40000
Destination IP Address,40000
Source Port,29761
Destination Port,29895
Protocol,3
Packet Length,1437
Attack Type,3
Severity Level,3
Action Taken,3


In [32]:
# Buat Mapping untuk Elasticsearch


# Siapkan nama field yang lebih praktis untuk Elasticsearch dan Kibana
rename_columns = {
    "Timestamp": "timestamp",
    "Source IP Address": "source_ip",
    "Destination IP Address": "destination_ip",
    "Source Port": "source_port",
    "Destination Port": "destination_port",
    "Protocol": "protocol",
    "Packet Length": "packet_length",
    "Attack Type": "attack_type",
    "Severity Level": "severity_level",
    "Action Taken": "action_taken",
    "Anomaly Scores": "anomaly_score",
    "Attack Signature": "attack_signature",
    "Packet Type": "packet_type",
    "Traffic Type": "traffic_type",
    "Network Segment": "network_segment",
    "Log Source": "log_source"
}

df_es = df_index.rename(columns=rename_columns).copy()

# Definisikan mapping Elasticsearch
index_mapping = {
    "dynamic": "strict",
    "properties": {
        "timestamp": {"type": "date"},
        "source_ip": {"type": "ip"},
        "destination_ip": {"type": "ip"},
        "source_port": {"type": "integer"},
        "destination_port": {"type": "integer"},
        "protocol": {"type": "keyword"},
        "packet_length": {"type": "integer"},
        "attack_type": {"type": "keyword"},
        "severity_level": {"type": "keyword"},
        "action_taken": {"type": "keyword"},
        "anomaly_score": {"type": "float"},
        "attack_signature": {"type": "keyword"},
        "packet_type": {"type": "keyword"},
        "traffic_type": {"type": "keyword"},
        "network_segment": {"type": "keyword"},
        "log_source": {"type": "keyword"}
    }
}

print("df_es shape:", df_es.shape)
print("Jumlah kolom:", len(df_es.columns))
print("Kolom:", df_es.columns.tolist())
print("Total missing values:", df_es.isna().sum().sum())
print("\nMapping Elasticsearch:")
print(index_mapping)


df_es shape: (40000, 16)
Jumlah kolom: 16
Kolom: ['timestamp', 'source_ip', 'destination_ip', 'source_port', 'destination_port', 'protocol', 'packet_length', 'attack_type', 'severity_level', 'action_taken', 'anomaly_score', 'attack_signature', 'packet_type', 'traffic_type', 'network_segment', 'log_source']
Total missing values: 0

Mapping Elasticsearch:
{'dynamic': 'strict', 'properties': {'timestamp': {'type': 'date'}, 'source_ip': {'type': 'ip'}, 'destination_ip': {'type': 'ip'}, 'source_port': {'type': 'integer'}, 'destination_port': {'type': 'integer'}, 'protocol': {'type': 'keyword'}, 'packet_length': {'type': 'integer'}, 'attack_type': {'type': 'keyword'}, 'severity_level': {'type': 'keyword'}, 'action_taken': {'type': 'keyword'}, 'anomaly_score': {'type': 'float'}, 'attack_signature': {'type': 'keyword'}, 'packet_type': {'type': 'keyword'}, 'traffic_type': {'type': 'keyword'}, 'network_segment': {'type': 'keyword'}, 'log_source': {'type': 'keyword'}}}


Hasilnya sudah sesuai. df_es memiliki 40.000 baris, 16 kolom, dan 0 missing values. Nama kolom juga sudah konsisten dalam snake_case, dan mapping sudah mendefinisikan tipe data yang sesuai dengan tujuan dashboard.

In [33]:
import requests

ELASTIC_URL = "https://my-observability-project-cbfcba.ingest.us-central1.gcp.elastic.cloud:443/_es"

response = requests.get(ELASTIC_URL)

print("Status code:", response.status_code)
print(response.text[:500])

Status code: 200
{"name":"managed-inputs","cluster_name":"elasticsearch","cluster_uuid":"cbfcba7002aa4774a96b5d2a30ab5914","version":{"number":"9.0.0","build_flavor":"default","build_type":"docker","build_hash":"0000000000000000000000000000000000000000","build_date":"2025-04-07T00:00:00.000Z","build_snapshot":false,"lucene_version":"10.1.0","minimum_wire_compatibility_version":"8.18.0","minimum_index_compatibility_version":"8.0.0"},"tagline":"You Know, for Search"}



Status code 200 berarti Google Colab berhasil terhubung ke Elasticsearch Cloud

# C. Koneksi ke ELasticsearch

In [51]:
from google.colab import userdata
import requests

ELASTIC_URL = "https://my-observability-project-cbfcba.es.us-central1.gcp.elastic.cloud:443"

# Ambil API key terbaru dari Colab Secrets
API_KEY = userdata.get("ELASTIC_API_KEY")

# Buat ulang headers menggunakan API key terbaru
headers = {
    "Authorization": f"ApiKey {API_KEY}"
}

# Periksa izin menulis
response = requests.get(
    f"{ELASTIC_URL}/_security/user/_has_privileges",
    headers=headers,
    json={
        "index": [
            {
                "names": ["security-events-test"],
                "privileges": ["create_doc", "create", "index"]
            }
        ]
    },
    timeout=30
)

print("Status code:", response.status_code)
print(response.text)

Status code: 200
{"username":"1902410597","has_all_requested":true,"cluster":{},"index":{"security-events-test":{"create":true,"create_doc":true,"index":true}},"application":{}}


Google Colab berhasil terhubung ke Elasticsearch Cloud menggunakan API key

In [52]:
# Periksa index security-events-test

response = requests.get(
    f"{ELASTIC_URL}/security-events-test",
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print(response.text[:1000])

Status code: 403
{"error":{"root_cause":[{"type":"security_exception","reason":"action [indices:admin/get] is unauthorized for API key id [anxpd6EB3NOizGNGZtUd] of user [1902410597] on indices [security-events-test], this action is granted by the index privileges [view_index_metadata,manage,all]"}],"type":"security_exception","reason":"action [indices:admin/get] is unauthorized for API key id [anxpd6EB3NOizGNGZtUd] of user [1902410597] on indices [security-events-test], this action is granted by the index privileges [view_index_metadata,manage,all]"},"status":403}


In [38]:
%pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [elasticsearch]


In [53]:
# Upload 1 baris sebagai percobaan

from elasticsearch import Elasticsearch
from google.colab import userdata

ELASTIC_URL = "https://my-observability-project-cbfcba.es.us-central1.gcp.elastic.cloud:443"
API_KEY = userdata.get("ELASTIC_API_KEY")

es = Elasticsearch(
    ELASTIC_URL,
    api_key=API_KEY
)

# Ambil satu baris dari dataframe yang sudah disiapkan
sample_doc = df_es.iloc[0].to_dict()

# Upload satu dokumen percobaan
result = es.index(
    index="security-events-test",
    document=sample_doc
)

print("Upload berhasil!")
print("Document ID:", result["_id"])
print("Result:", result["result"])

Upload berhasil!
Document ID: vnxrd6EB3NOizGNGJ9W2
Result: created


## Upload 40.000 Document


In [54]:
#Upload 39999 Document karena 1 document sudah di upload sebelumnya

from elasticsearch.helpers import bulk

def generate_actions():
    for record in df_es.iloc[1:].to_dict(orient="records"):
        yield {
            "_index": "security-events-test",
            "_source": record
        }

success, errors = bulk(
    es,
    generate_actions(),
    chunk_size=500,
    raise_on_error=False
)

print("Dokumen berhasil di-upload:", success)
print("Dokumen gagal:", len(errors))


Dokumen berhasil di-upload: 39999
Dokumen gagal: 0


In [55]:
print("Tanggal paling awal:", df_es["timestamp"].min())
print("Tanggal paling akhir:", df_es["timestamp"].max())

Tanggal paling awal: 2020-01-01 00:43:27+00:00
Tanggal paling akhir: 2023-10-11 19:34:23+00:00


In [56]:
print("Jumlah IP sumber unik:", df_es["source_ip"].nunique())
print("\n10 IP sumber yang paling sering muncul:")
print(df_es["source_ip"].value_counts().head(10))

Jumlah IP sumber unik: 40000

10 IP sumber yang paling sering muncul:
source_ip
138.156.5.40       1
24.150.253.66      1
188.133.58.141     1
113.228.203.251    1
209.70.190.115     1
122.163.144.175    1
24.231.154.145     1
99.127.94.207      1
134.187.11.18      1
99.194.235.221     1
Name: count, dtype: int64
